[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_methods/06_numerical_integration_quadrature/first_principles.ipynb)

# Topic 06: Numerical Integration (Quadrature)

## 1. First-Principles Intuition & Motivation

Almost every integral that matters lacks a closed form. There is no elementary antiderivative for $e^{-x^2}$ (the Gaussian CDF), for $\sqrt{1 - k^2\sin^2\theta}$ (the pendulum period and the arc length of an ellipse), for $\frac{x^3}{e^{x}-1}$ (the Stefan–Boltzmann law), or for the posterior normalizing constant of essentially any Bayesian model. Liouville's theorem makes this a *theorem*, not a failure of ingenuity: the class of elementary functions is not closed under integration.

So integrals must be *computed*, and every computational scheme has the same shape:

$$
\int_a^b f(x)\,dx \;\approx\; \sum_{i=1}^{n} w_i\,f(x_i).
$$

A **quadrature rule** is nothing more than a list of nodes $x_i$ and weights $w_i$. The entire theory is about how to choose them, and how to bound the error that results.

**The classical route: fix the nodes, interpolate, integrate.** Choose nodes (traditionally equispaced), pass the unique interpolating polynomial through the samples (Topic 04), and integrate that polynomial exactly. The weights come out as $w_i = \int_a^b L_i(x)\,dx$ and the error is inherited directly from the interpolation error. Two nodes give the trapezoidal rule; three give Simpson's; the family is Newton–Cotes.

### Two ideas that organize everything

**Idea 1 — degree of exactness.** A rule's quality is measured by the largest $m$ such that it integrates every polynomial of degree $\le m$ exactly. Trapezoid: $m = 1$. Simpson: $m = 3$ — *not* $2$, even though it fits a parabola, because the cubic error term integrates to zero by symmetry. That free extra degree is the first hint that **symmetry buys accuracy**, exactly as it did for central differences in Topic 05.

**Idea 2 — free the nodes.** With $n$ nodes and $n$ weights there are $2n$ free parameters, so one might hope to match $2n$ conditions and reach degree of exactness $2n - 1$. Gauss proved this optimum is attainable, and attainable in exactly one way: the nodes must be the roots of the degree-$n$ orthogonal polynomial for the weight function. Two Gauss–Legendre nodes integrate every cubic exactly; three integrate every quintic. Compare Simpson, which needs three nodes for degree $3$.

**The two failure modes to respect.**

- **High-order Newton–Cotes is unstable.** From $n = 8$ onward the weights change sign and grow without bound (the Runge phenomenon of Topic 04, integrated). The cure is never "raise the order" but "subdivide": composite rules apply a low-order rule on many small panels, converting a local $O(h^{k+1})$ error into a global $O(h^{k})$ one.
- **Deterministic quadrature dies in high dimension.** A tensor product of $m$-point rules in $d$ dimensions costs $N = m^{d}$ evaluations and converges like $N^{-k/d}$. At $d = 100$ this is hopeless for any $k$. Monte Carlo's $O(N^{-1/2})$ looks terrible in 1-D and is unbeatable at $d = 100$, because its rate does not depend on $d$ at all.

**One structural remark.** Integration is *smoothing*: an error $\delta$ in $f$ perturbs $\int f$ by at most $(b-a)\delta$. Quadrature is therefore perfectly well conditioned — the exact opposite of differentiation, where the error is amplified by $1/h$. You may integrate noisy data safely; you may never differentiate it naively.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition 1 (Quadrature rule).** A rule is a pair of finite sequences $(x_i, w_i)_{i=1}^{n}$ defining $Q_n[f] = \sum_{i=1}^n w_i f(x_i)$, intended to approximate $I[f] = \int_a^b f(x)\rho(x)\,dx$ for a fixed non-negative weight function $\rho$ (usually $\rho \equiv 1$). The error functional is $E_n[f] = I[f] - Q_n[f]$.

**Definition 2 (Degree of exactness).** $Q_n$ has degree of exactness $m$ if $E_n[p] = 0$ for every $p \in \mathbb{P}_m$ and $E_n[q] \neq 0$ for some $q \in \mathbb{P}_{m+1}$. By linearity it suffices to check the monomials $1, x, \ldots, x^{m}$.

**Definition 3 (Interpolatory rule).** $Q_n$ is *interpolatory* on its nodes if $w_i = \int_a^b L_i(x)\rho(x)\,dx$, where $L_i$ is the Lagrange basis for $x_1, \ldots, x_n$ — equivalently, if $Q_n[f] = I[\Pi_{n-1}f]$.

**Definition 4 (Newton–Cotes and composite rules).** A *closed Newton–Cotes* rule of order $n$ is the interpolatory rule on $n+1$ equispaced nodes including the endpoints. A *composite* rule partitions $[a,b]$ into $N$ panels of width $h = (b-a)/N$ and applies a fixed rule on each panel.

**Definition 5 (Orthogonal polynomials).** For a weight $\rho \ge 0$ on $(a,b)$ with finite moments, the sequence $\{p_k\}$ with $\deg p_k = k$ and $\int_a^b p_j p_k \rho\,dx = 0$ for $j \neq k$ is unique up to normalization; it satisfies a three-term recurrence $p_{k+1} = (x - \alpha_k)p_k - \beta_k p_{k-1}$, and all $k$ roots of $p_k$ are **real, simple, and interior** to $(a,b)$.

**Definition 6 (Gaussian quadrature).** The $n$-point Gauss rule for weight $\rho$ takes the nodes to be the roots of $p_n$ and the weights to be interpolatory, $w_i = \int_a^b L_i\rho\,dx$.

**Definition 7 (Monte Carlo estimator and star discrepancy).** For $X_1,\ldots,X_N \overset{\text{iid}}{\sim} \mathrm{Unif}(\Omega)$ with $\lvert\Omega\rvert$ the volume, $\hat I_N = \frac{\lvert\Omega\rvert}{N}\sum_k f(X_k)$. For a deterministic point set $P$, the star discrepancy is $D_N^{\ast}(P) = \sup_{B}\left\lvert \frac{\#(P\cap B)}{N} - \mathrm{vol}(B) \right\rvert$ over axis-aligned boxes $B$ anchored at the origin.

**Theorem 1 (Interpolatory $\iff$ exactness $\ge n-1$).** An $n$-node rule has degree of exactness at least $n-1$ if and only if it is interpolatory on those nodes. In particular the weights are uniquely determined by the nodes, via the Vandermonde moment system $\sum_i w_i x_i^{\,j} = \int_a^b x^{j}\rho\,dx$, $j = 0,\ldots,n-1$.

**Theorem 2 (Basic Newton–Cotes errors).** For $f$ smooth enough on $[a,b]$, with $h = b - a$:

$$
\text{Midpoint: } \int_a^b f = (b-a)f\!\left(\tfrac{a+b}{2}\right) + \frac{(b-a)^{3}}{24}f''(\xi),
$$

$$
\text{Trapezoid: } \int_a^b f = \frac{b-a}{2}\bigl(f(a)+f(b)\bigr) - \frac{(b-a)^{3}}{12}f''(\xi),
$$

$$
\text{Simpson: } \int_a^b f = \frac{b-a}{6}\left( f(a) + 4f\!\left(\tfrac{a+b}{2}\right) + f(b) \right) - \frac{(b-a)^{5}}{2880}f^{(4)}(\xi).
$$

Degrees of exactness: $1$, $1$, $3$ respectively. The midpoint error is **half** the trapezoid error and of the opposite sign.

**Theorem 3 (Composite rules).** With $N$ panels of width $h = (b-a)/N$:

$$
E^{\mathrm{comp}}_{\mathrm{mid}} = \frac{(b-a)h^{2}}{24}f''(\xi), \quad E^{\mathrm{comp}}_{\mathrm{trap}} = -\frac{(b-a)h^{2}}{12}f''(\xi), \quad E^{\mathrm{comp}}_{\mathrm{Simp}} = -\frac{(b-a)h^{4}}{180}f^{(4)}(\xi).
$$

One power of $h$ is always lost relative to the single-panel error, because there are $(b-a)/h$ panels.

**Theorem 4 (Euler–Maclaurin).** For $f \in C^{2m+2}[a,b]$ and the composite trapezoid rule $T(h)$,

$$
T(h) - \int_a^b f = \sum_{k=1}^{m}\frac{B_{2k}}{(2k)!}h^{2k}\left[ f^{(2k-1)}(b) - f^{(2k-1)}(a) \right] + O\!\left(h^{2m+2}\right),
$$

with Bernoulli numbers $B_2 = \tfrac16$, $B_4 = -\tfrac1{30}$, …. The expansion contains **only even powers of $h$** — the hypothesis that makes Romberg integration jump two orders per extrapolation. If $f$ is periodic with period $b - a$ (or all odd derivatives match at the endpoints), every term vanishes and the trapezoid rule converges **faster than any power of $h$**.

**Theorem 5 (Gauss: maximal degree of exactness).** No $n$-point rule has degree of exactness $2n$. The $n$-point Gauss rule for weight $\rho$ attains degree of exactness exactly $2n-1$, and it is the unique such rule. Its error is

$$
E_n[f] = \frac{f^{(2n)}(\xi)}{(2n)!}\int_a^b \prod_{i=1}^{n}(x-x_i)^{2}\,\rho(x)\,dx ,
$$

which for Gauss–Legendre on $[a,b]$ becomes $\frac{(b-a)^{2n+1}(n!)^{4}}{(2n+1)\bigl[(2n)!\bigr]^{3}}f^{(2n)}(\xi)$.

**Theorem 6 (Positivity and convergence).** All Gauss weights are strictly positive, $w_i = \int_a^b L_i^{2}\rho\,dx \gt 0$, and $\sum_i w_i = \int_a^b \rho\,dx$. Consequently (Stieltjes) $Q_n[f] \to I[f]$ for every $f \in C[a,b]$, and the rule is numerically stable: rounding errors in the samples are not amplified.

**Theorem 7 (Monte Carlo and quasi-Monte Carlo rates).** If $\sigma^{2} = \operatorname{Var}f(X) \lt \infty$ then $\mathbb{E}[\hat I_N] = I$ and

$$
\mathrm{RMSE}(\hat I_N) = \frac{\lvert\Omega\rvert\,\sigma}{\sqrt{N}} = O\!\left(N^{-1/2}\right) \quad \text{independently of the dimension } d .
$$

For a low-discrepancy sequence, the Koksma–Hlawka inequality gives $\lvert \hat I_N - I\rvert \le V_{\mathrm{HK}}(f)\,D_N^{\ast}$ with $D_N^{\ast} = O\!\left(\frac{(\log N)^{d}}{N}\right)$, i.e. nearly $O(N^{-1})$ for moderate $d$.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 1 — Trapezoid and midpoint errors from the interpolation error

**Trapezoid.** The rule integrates the linear interpolant through $(a, f(a))$ and $(b, f(b))$. By Topic 04's error theorem, $f(x) - p_1(x) = \frac{f''(\xi_x)}{2}(x-a)(x-b)$. Integrating,

$$
E = \int_a^b \frac{f''(\xi_x)}{2}(x-a)(x-b)\,dx .
$$

The factor $(x-a)(x-b)$ does **not change sign** on $[a,b]$ (it is $\le 0$ throughout), so the **mean value theorem for integrals** applies: there is $\xi\in(a,b)$ with

$$
E = \frac{f''(\xi)}{2}\int_a^b (x-a)(x-b)\,dx = \frac{f''(\xi)}{2}\cdot\left(-\frac{(b-a)^{3}}{6}\right) = -\frac{(b-a)^{3}}{12}f''(\xi). \qquad \blacksquare
$$

(The integral $\int_a^b (x-a)(x-b)dx = -\frac{(b-a)^3}{6}$ follows from substituting $x = a + t(b-a)$ and using $\int_0^1 t(t-1)dt = -\tfrac16$.)

**Midpoint — a subtler argument.** The rule uses only $m = \frac{a+b}{2}$, so naively it integrates the *constant* interpolant and one expects $O(h^2)$. The direct interpolation argument fails because $(x - m)$ changes sign. Instead Taylor-expand about $m$:

$$
f(x) = f(m) + (x-m)f'(m) + \frac{(x-m)^{2}}{2}f''(\xi_x).
$$

Integrate over $[a,b]$. The linear term integrates to **zero by symmetry** — $\int_a^b (x-m)dx = 0$ — which is the source of the extra order. The quadratic term has $(x-m)^2 \ge 0$ of one sign, so the mean value theorem gives

$$
\int_a^b f = (b-a)f(m) + \frac{f''(\xi)}{2}\int_a^b (x-m)^{2}dx = (b-a)f(m) + \frac{f''(\xi)}{2}\cdot\frac{(b-a)^{3}}{12} = (b-a)f(m) + \frac{(b-a)^{3}}{24}f''(\xi). \qquad \blacksquare
$$

**Consequences.** The midpoint rule uses **one** evaluation and has **half** the error of the two-evaluation trapezoid rule — a $4\times$ efficiency advantage per evaluation — and errors of opposite sign, so the two bracket the true value when $f''$ has constant sign. Eliminating $f''$ between them gives $\frac{2M + T}{3}$, which is exactly Simpson's rule: Simpson is the Richardson extrapolation of midpoint against trapezoid.

### Proof 2 — Simpson's rule and the free extra degree

**Claim.** $\int_a^b f = \frac{b-a}{6}\bigl(f(a) + 4f(m) + f(b)\bigr) - \frac{(b-a)^{5}}{2880}f^{(4)}(\xi)$, with degree of exactness $3$.

**Step 1 — the weights.** Interpolate at $a, m, b$ and integrate the parabola. With $h = \frac{b-a}{2}$ and the substitution $x = m + th$, $t \in [-1,1]$, the Lagrange basis becomes $\frac{t(t-1)}{2}, (1-t^2), \frac{t(t+1)}{2}$ and

$$
\int_{-1}^{1}\frac{t(t-1)}{2}dt = \frac13, \qquad \int_{-1}^{1}(1-t^{2})dt = \frac43, \qquad \int_{-1}^{1}\frac{t(t+1)}{2}dt = \frac13 .
$$

Multiplying by $h$: $Q = \frac{h}{3}\bigl(f(a) + 4f(m) + f(b)\bigr) = \frac{b-a}{6}\bigl(f(a)+4f(m)+f(b)\bigr)$.

**Step 2 — degree of exactness is 3, not 2.** Exactness on $\mathbb{P}_2$ is automatic (the rule integrates the exact interpolant). For $f(x) = (x-m)^{3}$: the true integral is $\int_{-h}^{h}s^{3}ds = 0$ by oddness, and the rule gives $\frac{h}{3}\bigl((-h)^3 + 4\cdot 0 + h^3\bigr) = 0$. **Exact.** Hence $m \ge 3$. For $f(x) = (x-m)^4$ the true value is $\frac{2h^{5}}{5}$ while the rule gives $\frac{h}{3}(2h^{4}) = \frac{2h^{5}}{3} \neq \frac{2h^5}{5}$, so $m = 3$ exactly.

**Step 3 — the error constant by the method of exact monomials.** Since the rule is exact on $\mathbb{P}_3$ and the error functional is linear, expand $f$ about $m$ to third order with remainder; only the quartic term survives. Applying $E$ to $\frac{(x-m)^4}{4!}f^{(4)}$:

$$
E[f] = \frac{f^{(4)}(\xi)}{4!}\,E\bigl[(x-m)^{4}\bigr] = \frac{f^{(4)}(\xi)}{24}\left( \frac{2h^{5}}{5} - \frac{2h^{5}}{3} \right) = \frac{f^{(4)}(\xi)}{24}\cdot\left(-\frac{4h^{5}}{15}\right) = -\frac{h^{5}}{90}f^{(4)}(\xi).
$$

With $h = \frac{b-a}{2}$, $h^{5} = \frac{(b-a)^5}{32}$ and $\frac{1}{90\cdot32} = \frac{1}{2880}$:

$$
E = -\frac{(b-a)^{5}}{2880}f^{(4)}(\xi). \qquad \blacksquare
$$

**The general symmetry principle.** A symmetric rule with an **odd** number of nodes gains one degree of exactness for free, because the next monomial in the expansion is odd about the centre and integrates to zero on both sides. This is the same parity argument that makes central differences second-order rather than first — quadrature and differentiation are the same interpolation theory read in two directions.

### Proof 3 — Composite rules lose exactly one power of $h$

**Claim.** (Theorem 3.) The composite trapezoid rule on $N$ panels of width $h$ satisfies $E = -\frac{(b-a)h^{2}}{12}f''(\xi)$; composite Simpson satisfies $E = -\frac{(b-a)h^{4}}{180}f^{(4)}(\xi)$.

**Proof (trapezoid).** Apply the single-panel result on each $[x_{j}, x_{j+1}]$, $j = 0, \ldots, N-1$:

$$
E = \sum_{j=0}^{N-1}\left( -\frac{h^{3}}{12}f''(\xi_j) \right) = -\frac{h^{3}}{12}\sum_{j=0}^{N-1}f''(\xi_j) = -\frac{h^{3}}{12}\cdot N \cdot \underbrace{\frac{1}{N}\sum_j f''(\xi_j)}_{= f''(\xi) \text{ by IVT}} .
$$

The average of $N$ values of the continuous function $f''$ lies between its min and max, so by the intermediate value theorem it equals $f''(\xi)$ for some $\xi\in(a,b)$. Using $Nh = b-a$:

$$
E = -\frac{h^{2}(Nh)}{12}f''(\xi) = -\frac{(b-a)h^{2}}{12}f''(\xi). \qquad \blacksquare
$$

**Simpson.** Identical, with panels of width $2h$ carrying local error $-\frac{(2h)^5}{2880}f^{(4)} = -\frac{h^{5}}{90}f^{(4)}$ and $N/2$ panels:

$$
E = -\frac{h^{5}}{90}\cdot\frac{N}{2}f^{(4)}(\xi) = -\frac{h^{4}(Nh)}{180}f^{(4)}(\xi) = -\frac{(b-a)h^{4}}{180}f^{(4)}(\xi). \qquad \blacksquare
$$

**The accounting rule, stated once.** Local error $O(h^{k+1})$ $\times$ $O(1/h)$ panels $=$ global error $O(h^{k})$. This one line explains every table of composite orders and is the exact analogue of the local/global truncation error relationship in ODE solvers.

**Convergence in evaluations.** With $N \propto 1/h$ evaluations, composite trapezoid gives $E = O(N^{-2})$ and composite Simpson $E = O(N^{-4})$. Gauss–Legendre with $n$ nodes gives $E = O(\rho^{-2n})$ for analytic $f$ — *geometric*, not algebraic. That is the whole argument for Gauss on smooth integrands.

### Proof 4 — Gaussian quadrature: degree of exactness $2n-1$, and its optimality

**Claim.** (Theorem 5.) Let $p_n$ be the degree-$n$ orthogonal polynomial for weight $\rho$ on $(a,b)$, let $x_1,\ldots,x_n$ be its roots, and let the $w_i$ be interpolatory. Then $Q_n$ is exact for every $f \in \mathbb{P}_{2n-1}$, and no $n$-point rule is exact on $\mathbb{P}_{2n}$.

**Part A — exactness on $\mathbb{P}_{2n-1}$.** Let $f \in \mathbb{P}_{2n-1}$. Divide by $p_n$:

$$
f = q\,p_n + r, \qquad q, r \in \mathbb{P}_{n-1}
$$

(the quotient has degree $\le (2n-1) - n = n-1$, the remainder degree $\le n-1$). Now integrate:

$$
I[f] = \underbrace{\int_a^b q\,p_n\,\rho\,dx}_{=\,0} + \int_a^b r\,\rho\,dx = I[r],
$$

where the first integral vanishes **by orthogonality**: $q \in \mathbb{P}_{n-1}$ is a combination of $p_0,\ldots,p_{n-1}$, each orthogonal to $p_n$.

On the quadrature side, $p_n(x_i) = 0$ for every node, so

$$
Q_n[f] = \sum_i w_i\bigl(q(x_i)p_n(x_i) + r(x_i)\bigr) = \sum_i w_i r(x_i) = Q_n[r].
$$

Finally $Q_n[r] = I[r]$ because $r \in \mathbb{P}_{n-1}$ and the rule is interpolatory on $n$ nodes (Theorem 1). Chaining: $I[f] = I[r] = Q_n[r] = Q_n[f]$. $\blacksquare$

**Part B — $2n-1$ cannot be beaten.** Let $Q_n$ be *any* $n$-point rule with nodes $x_1,\ldots,x_n$ and consider

$$
f(x) = \prod_{i=1}^{n}(x - x_i)^{2} \in \mathbb{P}_{2n}, \qquad f \ge 0, \ f \not\equiv 0 .
$$

Then $Q_n[f] = \sum_i w_i f(x_i) = 0$ (every node is a double root), while

$$
I[f] = \int_a^b \prod_i (x-x_i)^2\,\rho(x)\,dx \gt 0
$$

since the integrand is non-negative, continuous, not identically zero, and $\rho \ge 0$ with positive mass. So $E_n[f] \neq 0$ and no $n$-point rule is exact on $\mathbb{P}_{2n}$. $\blacksquare$

**Part C — the nodes *must* be the orthogonal-polynomial roots.** Suppose an $n$-point rule is exact on $\mathbb{P}_{2n-1}$, and set $\omega(x) = \prod_i (x - x_i)$. For any $q \in \mathbb{P}_{n-1}$ the product $q\omega \in \mathbb{P}_{2n-1}$, so exactness gives

$$
\int_a^b q\,\omega\,\rho\,dx = \sum_i w_i\,q(x_i)\,\omega(x_i) = 0 .
$$

Thus the monic $\omega$ of degree $n$ is orthogonal to all of $\mathbb{P}_{n-1}$ — which characterizes $p_n$ up to normalization. Hence $\omega = p_n/(\text{lead})$ and the nodes are its roots. Uniqueness follows. $\blacksquare$

**The error term.** Let $H$ be the Hermite interpolant of $f$ at the $n$ nodes (matching values *and* derivatives), so $H \in \mathbb{P}_{2n-1}$ and, by Topic 04's Hermite error formula, $f - H = \frac{f^{(2n)}(\xi_x)}{(2n)!}\omega(x)^{2}$. Since the rule is exact on $\mathbb{P}_{2n-1}$ and $H$ agrees with $f$ at the nodes, $Q_n[f] = Q_n[H] = I[H]$, whence

$$
E_n[f] = I[f] - I[H] = \int_a^b \frac{f^{(2n)}(\xi_x)}{(2n)!}\,\omega(x)^{2}\rho(x)\,dx = \frac{f^{(2n)}(\xi)}{(2n)!}\int_a^b \omega^{2}\rho\,dx,
$$

the mean value theorem applying because $\omega^2\rho \ge 0$. $\blacksquare$

### Proof 5 — Gauss weights are positive, and why that matters

**Claim.** (Theorem 6.) $w_i \gt 0$ for every $i$, and $\sum_i w_i = \int_a^b \rho\,dx$.

**Proof.** Fix $i$ and take the test function $f = L_i^{2}$, the square of the $i$-th Lagrange basis polynomial. Then $\deg f = 2(n-1) = 2n - 2 \le 2n-1$, so the Gauss rule integrates it **exactly**:

$$
0 \lt \int_a^b L_i(x)^{2}\rho(x)\,dx = Q_n[L_i^{2}] = \sum_{j} w_j L_i(x_j)^{2} = \sum_j w_j \delta_{ij}^{2} = w_i .
$$

The left inequality is strict because $L_i^2 \ge 0$ is continuous and equals $1$ at $x_i$. The sum identity follows from exactness on $f \equiv 1 \in \mathbb{P}_0$. $\blacksquare$

**Why positivity is decisive.**

1. **Numerical stability.** If each $f(x_i)$ carries an absolute error $\le \delta$, the computed rule has error at most $\sum_i \lvert w_i\rvert \delta = \bigl(\int\rho\bigr)\delta$ — bounded independently of $n$. For a rule with mixed-sign weights, $\sum_i \lvert w_i \rvert$ can grow without bound and the rule amplifies noise. High-order Newton–Cotes rules do exactly this: at $n = 8$ the weights change sign, and $\sum\lvert w_i\rvert \to \infty$.

2. **Convergence for all continuous $f$ (Stieltjes' theorem).** Positive weights plus $\sum_i w_i = \int\rho$ plus exactness of increasing degree gives, via Weierstrass approximation: for any $\epsilon$ pick $p \in \mathbb{P}_m$ with $\lVert f - p\rVert_\infty \lt \epsilon$; then for $2n-1 \ge m$,

$$
\lvert I[f] - Q_n[f] \rvert \le \lvert I[f-p]\rvert + \lvert Q_n[f-p]\rvert \le \epsilon\int\rho + \epsilon\sum_i \lvert w_i\rvert = 2\epsilon\int_a^b \rho\,dx .
$$

Hence $Q_n[f] \to I[f]$ for **every** continuous $f$ — no smoothness required. This is precisely the guarantee that equispaced interpolation (and therefore high-order Newton–Cotes) fails to provide.

3. **Probabilistic interpretation.** Normalized Gauss weights form a discrete probability distribution matching the first $2n-1$ moments of $\rho$. Gauss–Hermite quadrature is thus an optimal $n$-point discrete approximation of a Gaussian — the basis of unscented Kalman filters, sigma-point methods, and deterministic ELBO evaluation.

### Proof 6 — Monte Carlo, the curse of dimensionality, and QMC

**Claim.** (Theorem 7.) $\hat I_N = \frac{\lvert\Omega\rvert}{N}\sum_k f(X_k)$ is unbiased with RMSE $\frac{\lvert\Omega\rvert\sigma}{\sqrt N}$, independent of $d$.

**Proof.** With $X_k$ iid uniform on $\Omega$, $\mathbb{E}[f(X_k)] = \frac{1}{\lvert\Omega\rvert}\int_\Omega f$, so

$$
\mathbb{E}[\hat I_N] = \frac{\lvert\Omega\rvert}{N}\cdot N\cdot\frac{I}{\lvert\Omega\rvert} = I .
$$

By independence the variances add:

$$
\operatorname{Var}(\hat I_N) = \frac{\lvert\Omega\rvert^{2}}{N^{2}}\sum_{k=1}^{N}\operatorname{Var}\bigl(f(X_k)\bigr) = \frac{\lvert\Omega\rvert^{2}\sigma^{2}}{N}, \qquad \mathrm{RMSE} = \frac{\lvert\Omega\rvert\sigma}{\sqrt N}. \qquad \blacksquare
$$

**Nothing in this argument mentions $d$.** The dimension enters only through $\sigma$, which depends on the integrand, not on the geometry of the sampling.

**The curse, quantified.** A tensor-product rule of order $k$ with $m$ points per axis uses $N = m^{d}$ evaluations and has error

$$
E = O(m^{-k}) = O\!\left( N^{-k/d} \right).
$$

Crossover with Monte Carlo's $N^{-1/2}$ occurs at $d = 2k$:

| $d$ | Simpson tensor grid ($k=4$) | Gauss tensor grid ($k = 2n$, effectively large) | Monte Carlo |
| :--- | :--- | :--- | :--- |
| 1 | $N^{-4}$ | geometric | $N^{-1/2}$ |
| 4 | $N^{-1}$ | $N^{-\text{large}/4}$ | $N^{-1/2}$ |
| 8 | $N^{-1/2}$ (tie) | — | $N^{-1/2}$ |
| 100 | $N^{-0.04}$ | hopeless | $N^{-1/2}$ |

At $d = 100$ with just $m = 3$ points per axis, $N = 3^{100} \approx 5\times10^{47}$ evaluations — more than the number of atoms in the Earth. Monte Carlo with $N = 10^{6}$ gives three-digit accuracy for any $d$.

**The cost of the MC rate.** $\mathrm{RMSE}\propto N^{-1/2}$ means **one extra digit costs $100\times$ the work**. Hence the entire industry of variance reduction: control variates ($\operatorname{Var}$ reduced by $1-\rho^2$), importance sampling (sample where $\lvert f\rvert p$ is large), antithetic variates (cancel the odd part), stratification, and Rao–Blackwellization.

**Quasi-Monte Carlo.** Replace randomness by a deterministic low-discrepancy sequence (Sobol', Halton, lattice rules). The **Koksma–Hlawka** inequality bounds the error by a product of a function property and a point-set property:

$$
\left\lvert \frac{1}{N}\sum_k f(u_k) - \int_{[0,1]^d} f \right\rvert \le V_{\mathrm{HK}}(f)\, D_N^{\ast}(u_1,\ldots,u_N),
$$

where $V_{\mathrm{HK}}$ is the Hardy–Krause variation. Sobol' points achieve $D_N^{\ast} = O\!\left(\frac{(\log N)^{d}}{N}\right)$, so QMC converges nearly as $N^{-1}$ — a full power better than MC. Two caveats: the constant hides $(\log N)^{d}$, which is only benign when the *effective* dimension is small (as it usually is in finance and in many ML integrals), and $V_{\mathrm{HK}}(f)$ is infinite for discontinuous integrands, where QMC loses its advantage. Randomized QMC (scrambled Sobol') restores unbiasedness and an error estimate while keeping the improved rate.

## 4. Computational & Algorithmic Insights

### Romberg integration

Euler–Maclaurin (Theorem 4) says the composite trapezoid error is a series in $h^{2}$ alone. Richardson extrapolation therefore climbs two orders per level. Set $R_{k,0} = T(h_k)$ with $h_k = (b-a)/2^{k}$, and

$$
R_{k,m} = \frac{4^{m}R_{k,m-1} - R_{k-1,m-1}}{4^{m}-1}, \qquad R_{k,m} = I + O\!\left(h_k^{2m+2}\right).
$$

Column $0$ is trapezoid ($O(h^2)$), column $1$ is **exactly composite Simpson** ($O(h^4)$), column $2$ is Boole's rule ($O(h^6)$), and so on. The halved trapezoid sums are computed incrementally — $T(h/2) = \tfrac12 T(h) + \tfrac{h}{2}\sum_{\text{new midpoints}} f$ — so each new row costs only as many evaluations as it adds.

```python
import numpy as np

def romberg(f, a, b, levels=6):
    R = np.zeros((levels, levels))
    h = b - a
    R[0, 0] = 0.5 * h * (f(a) + f(b))
    for k in range(1, levels):
        h /= 2
        pts = a + h * np.arange(1, 2 ** k, 2)          # only the new midpoints
        R[k, 0] = 0.5 * R[k - 1, 0] + h * np.sum(f(pts))
        for m in range(1, k + 1):
            R[k, m] = (4**m * R[k, m - 1] - R[k - 1, m - 1]) / (4**m - 1)
    return R[levels - 1, levels - 1], R
```

The same caveat as in Topic 05 applies: extrapolation assumes the error expansion exists. An integrand with an endpoint singularity ($\sqrt{x}$ at $0$) has a $h^{3/2}$ term, Euler–Maclaurin breaks, and Romberg stalls — the cure is a variable transformation ($x = t^{2}$, or the double-exponential `tanh–sinh` transform) that restores smoothness.

### Adaptive quadrature

A fixed rule wastes evaluations on flat regions and under-resolves peaks. Adaptive quadrature estimates the local error and subdivides only where needed:

1. Compute $S(a,b)$ (a Simpson or Gauss–Kronrod estimate) on the interval.
2. Compute $S(a,m) + S(m,b)$ on the two halves.
3. **Error estimate** by Richardson: for Simpson, $\lvert S(a,b) - S(a,m) - S(m,b)\rvert / 15$ estimates the error of the refined value, because halving divides an $O(h^{4})$ error by $16$.
4. If the estimate is below the local tolerance, accept $S(a,m)+S(m,b)$ (optionally plus the extrapolated correction); otherwise recurse on each half with tolerance $\tau/2$.

```python
def adaptive_simpson(f, a, b, tol=1e-10, depth=50):
    def simp(a, b):
        m = 0.5 * (a + b)
        return (b - a) / 6.0 * (f(a) + 4 * f(m) + f(b))
    def rec(a, b, whole, tol, depth):
        m = 0.5 * (a + b)
        left, right = simp(a, m), simp(m, b)
        if depth <= 0 or abs(left + right - whole) <= 15 * tol:
            return left + right + (left + right - whole) / 15.0
        return (rec(a, m, left, tol / 2, depth - 1)
                + rec(m, b, right, tol / 2, depth - 1))
    return rec(a, b, simp(a, b), tol, depth)
```

**Gauss–Kronrod** is the professional version: the $(2n+1)$-point Kronrod extension **reuses all $n$ Gauss nodes** and adds $n+1$ more, giving a degree-$3n+1$ estimate whose difference from the Gauss value is a reliable error indicator at little extra cost. `scipy.integrate.quad` is QUADPACK's adaptive Gauss–Kronrod (G7–K15) with $\epsilon$-algorithm extrapolation for endpoint singularities.

**Where adaptivity fails.** A narrow spike that no initial node touches is invisible — the routine reports convergence on a function it never sampled. `quad` warns about this; the fix is to pass the known singular/peak locations via `points=[...]`, or to split the integral by hand.

### The Gauss families and how the nodes are computed

| Family | Weight $\rho(x)$ | Interval | Typical use |
| :--- | :--- | :--- | :--- |
| Gauss–Legendre | $1$ | $[-1,1]$ | general smooth integrands |
| Gauss–Chebyshev | $(1-x^{2})^{-1/2}$ | $[-1,1]$ | endpoint square-root singularities |
| Gauss–Hermite | $e^{-x^{2}}$ | $(-\infty,\infty)$ | Gaussian expectations, Bayesian inference |
| Gauss–Laguerre | $x^{\alpha}e^{-x}$ | $[0,\infty)$ | exponential/Gamma expectations, reliability |
| Gauss–Jacobi | $(1-x)^{\alpha}(1+x)^{\beta}$ | $[-1,1]$ | algebraic endpoint singularities |

Low-order Gauss–Legendre nodes and weights (on $[-1,1]$):

| $n$ | nodes | weights | exact to degree |
| :--- | :--- | :--- | :--- |
| 1 | $0$ | $2$ | 1 |
| 2 | $\pm 1/\sqrt3 = \pm0.5773503$ | $1, 1$ | 3 |
| 3 | $0, \pm\sqrt{3/5} = \pm0.7745967$ | $8/9, 5/9, 5/9$ | 5 |

**Golub–Welsch.** Do not solve the nonlinear moment equations (hopelessly ill conditioned). Instead form the symmetric tridiagonal **Jacobi matrix** $J$ from the three-term recurrence coefficients $\alpha_k, \beta_k$; then the nodes are the **eigenvalues** of $J$ and the weights are $w_i = \mu_0\,(v_i)_1^{2}$, the squared first components of the normalized eigenvectors, with $\mu_0 = \int\rho$. Cost: $O(n^{2})$, backward stable, and this is what `numpy.polynomial.legendre.leggauss` and `scipy.special.roots_hermite` implement.

**Clenshaw–Curtis, the modern rival.** Using Chebyshev nodes $x_k = \cos(k\pi/n)$ with weights computed by an FFT gives degree of exactness only $n$ (versus Gauss's $2n-1$), yet for most integrands it converges at essentially the same *observed* rate, costs $O(n\log n)$ instead of $O(n^{2})$, and its nodes are **nested** (reusable when refining). Trefethen's analysis explains the paradox: for functions of finite smoothness both rules converge at the same algebraic rate, and Gauss's factor-of-2 advantage in degree materializes only for analytic integrands.

### High-dimensional strategies and a decision table

| Regime | Method | Rate | Notes |
| :--- | :--- | :--- | :--- |
| $d = 1$, smooth/analytic | Gauss–Legendre or Clenshaw–Curtis | geometric | 10–20 nodes give machine precision |
| $d = 1$, endpoint singularity | tanh–sinh (double exponential), or Gauss–Jacobi | geometric | handles $x^{-1/2}$, $\log x$ |
| $d = 1$, peaked/irregular | adaptive Gauss–Kronrod (`quad`) | adaptive | pass known peak locations |
| $d = 1$, periodic | plain composite trapezoid | spectral | Euler–Maclaurin terms all vanish |
| $2 \le d \le 5$ | tensor Gauss, or sparse (Smolyak) grids | $O(N^{-k}(\log N)^{(d-1)(k+1)})$ | Smolyak beats full tensor products |
| $d \gtrsim 10$ | Monte Carlo + variance reduction | $O(N^{-1/2})$ | dimension free |
| $d \gtrsim 10$, low effective dimension | quasi-Monte Carlo (scrambled Sobol') | $\approx O(N^{-1})$ | needs bounded Hardy–Krause variation |
| Intractable target density | MCMC / HMC, or nested sampling | $O(N^{-1/2})$ with autocorrelation | for expectations under an unnormalized $p$ |

**Sparse (Smolyak) grids** deserve a note: instead of the full tensor product of 1-D rules, take a carefully signed combination of tensor products whose total order is bounded. The point count grows like $O(m(\log m)^{d-1})$ rather than $m^{d}$, buying several usable dimensions before Monte Carlo becomes necessary. This is the standard tool for uncertainty quantification with $d \sim 10$–$30$ and is the basis of polynomial-chaos expansions.

## 5. Real-World Physics & AI/ML Applications

**Statistical mechanics and radiation.** The Stefan–Boltzmann constant follows from $\int_0^\infty \frac{x^{3}}{e^{x}-1}dx = \frac{\pi^{4}}{15}$; the Debye model of heat capacity needs $\int_0^{\theta_D/T}\frac{x^{4}e^{x}}{(e^{x}-1)^{2}}dx$, which has no closed form and is evaluated by Gauss–Laguerre or adaptive quadrature in every solid-state code. Partition functions $Z = \int e^{-\beta H(q,p)}dq\,dp$ over $d \sim 10^{23}$ degrees of freedom are the original motivation for Monte Carlo (Metropolis et al., 1953).

**Orbital and structural mechanics.** The period of a finite-amplitude pendulum is $T = 4\sqrt{L/g}\,K(\sin\frac{\theta_0}{2})$ with $K$ the complete elliptic integral of the first kind — no elementary form, computed by the arithmetic–geometric mean or by Gauss–Chebyshev quadrature (whose weight matches the $(1-x^2)^{-1/2}$ endpoint behaviour exactly). Finite-element mass and stiffness matrices are assembled by Gauss–Legendre quadrature on each element; the choice of quadrature order is a design decision, since *under-integration* deliberately introduces hourglass modes and *over-integration* causes locking.

**Signal processing and spectra.** Fourier coefficients $\hat f(k) = \int_0^{2\pi}f(x)e^{-ikx}dx$ of a periodic $f$ are computed by the plain trapezoid rule — which, by the periodic case of Euler–Maclaurin, converges **spectrally**. The DFT/FFT is exactly the composite trapezoid rule applied to a periodic integrand, and its exponential accuracy is a quadrature theorem.

**Financial engineering.** Option prices are discounted expectations $e^{-rT}\mathbb{E}[\text{payoff}]$. In 1-D these are Gauss–Hermite or Fourier (Carr–Madan) integrals; for path-dependent and basket products with $d \sim 100$ time steps or assets, they are Monte Carlo, and quasi-Monte Carlo with Brownian-bridge construction — which concentrates the variance in the first few dimensions, keeping the *effective* dimension small — is standard practice.

**Machine learning applications.**

- **The ELBO and variational inference.** The evidence lower bound
$$
\mathcal{L}(\phi) = \mathbb{E}_{q_\phi(z)}\bigl[\log p(x, z) - \log q_\phi(z)\bigr]
$$
is an integral over the latent space. Three regimes: (i) **closed form** — for Gaussian $q$ and Gaussian prior the KL term is analytic, which is why VAEs use that pairing; (ii) **Gauss–Hermite** — for $d \lesssim 3$ latent dimensions (or a factorized $q$), the transformation $z = \mu + \sqrt2\,\sigma t$ turns $\mathbb{E}_{\mathcal N(\mu,\sigma^2)}[g]$ into $\frac{1}{\sqrt\pi}\sum_i w_i\, g(\mu + \sqrt2\sigma t_i)$ with $5$–$20$ nodes giving machine precision; this is exactly what GPflow and GPy use for non-Gaussian likelihoods in sparse GP classification; (iii) **reparameterized Monte Carlo** — for high-dimensional $z$, one or a few samples with the pathwise gradient, accepting $O(N^{-1/2})$ noise because the optimizer averages over minibatches anyway.

- **Gaussian processes with non-Gaussian likelihoods.** Predictive probabilities $\int \sigma(f)\,\mathcal{N}(f\mid\mu,\sigma^{2})\,df$ for GP classification, and the site updates in expectation propagation, are 1-D Gaussian integrals evaluated by Gauss–Hermite quadrature — the canonical "$2n-1$ exactness in a Gaussian weight" application. The probit likelihood even yields a closed form, which is exactly why probit is often preferred to logit in GP classification.

- **Bayesian model evidence.** $p(x) = \int p(x\mid\theta)p(\theta)d\theta$ is the normalizing constant that model comparison needs. In low dimension, adaptive quadrature; in moderate dimension, Laplace approximation (a Gaussian integral done exactly) or bridge/thermodynamic integration $\log Z = \int_0^1 \mathbb{E}_{p_\beta}[\log p(x\mid\theta)]\,d\beta$ — a 1-D quadrature over an inverse-temperature path where the integrand is itself an MCMC estimate, so a low-order composite rule with many $\beta$ values beats a high-order rule with few.

- **Diffusion models and neural ODEs.** Sampling from a diffusion model integrates a probability-flow ODE; likelihood evaluation integrates $\nabla\cdot f$ along the trajectory, estimated by the Hutchinson trace identity $\operatorname{tr}(A) = \mathbb{E}_{v}[v^{\mathsf T}Av]$ — a Monte Carlo quadrature in disguise, with the same $O(N^{-1/2})$ rate. Number-of-function-evaluations (NFE) benchmarks for samplers are literally quadrature-efficiency comparisons.

- **Attention approximations and kernel methods.** Random Fourier features approximate a shift-invariant kernel $k(x-y) = \int p(\omega)e^{i\omega^{\mathsf T}(x-y)}d\omega$ by Monte Carlo over $\omega$, with error $O(D^{-1/2})$ in the number of features; quasi-Monte Carlo and orthogonal random features (which are structured low-discrepancy point sets) provably improve the constant and the rate. Performer-style linear attention is the same construction applied to the softmax kernel.

- **Expected calibration and metrics.** Expected calibration error, AUC, and expected utility are integrals over a score distribution; their finite-sample estimators are quadrature rules (binned ECE is a composite midpoint rule, and its bias is the $O(h^2)$ term of Theorem 3, which is why ECE is sensitive to the bin count).

### Summary of key results

| Result | Statement |
| :--- | :--- |
| Interpolatory rule | $w_i = \int L_i \rho$; degree of exactness $\ge n-1$ |
| Midpoint | $(b-a)f(m) + \frac{(b-a)^{3}}{24}f''(\xi)$, degree 1 |
| Trapezoid | $\frac{b-a}{2}(f(a)+f(b)) - \frac{(b-a)^{3}}{12}f''(\xi)$, degree 1 |
| Simpson | $\frac{b-a}{6}(f(a)+4f(m)+f(b)) - \frac{(b-a)^{5}}{2880}f^{(4)}(\xi)$, degree 3 |
| Composite trapezoid | $-\frac{(b-a)h^{2}}{12}f''(\xi)$, $O(h^{2})$ |
| Composite Simpson | $-\frac{(b-a)h^{4}}{180}f^{(4)}(\xi)$, $O(h^{4})$ |
| Local-to-global | local $O(h^{k+1})$ $\times$ $O(1/h)$ panels $=$ global $O(h^{k})$ |
| Euler–Maclaurin | trapezoid error is a series in $h^{2}$; vanishes for periodic $f$ |
| Romberg | $R_{k,m} = \frac{4^{m}R_{k,m-1}-R_{k-1,m-1}}{4^{m}-1}$, order $2m+2$ |
| Gauss exactness | $2n-1$, maximal; nodes $=$ roots of $p_n$; unique |
| Gauss weights | $w_i = \int L_i^{2}\rho \gt 0$; guarantees stability and convergence |
| Gauss error | $\frac{f^{(2n)}(\xi)}{(2n)!}\int \omega^{2}\rho$ |
| Curse of dimensionality | tensor rule of order $k$: $O(N^{-k/d})$ |
| Monte Carlo | RMSE $= \sigma \lvert\Omega\rvert N^{-1/2}$, independent of $d$ |
| Quasi-Monte Carlo | Koksma–Hlawka: error $\le V_{\mathrm{HK}}(f)D_N^{\ast}$, $D_N^{\ast} = O((\log N)^{d}/N)$ |

## 6. Canonical Literature Mapping & References

| Concept | Canonical source | Location |
| :--- | :--- | :--- |
| Newton–Cotes, composite rules, error terms | Burden & Faires, *Numerical Analysis* | Ch. 4.3–4.4 |
| Romberg integration | Burden & Faires | Ch. 4.5 |
| Adaptive quadrature | Burden & Faires; Press et al. | Ch. 4.6; Ch. 4.7 |
| Gaussian quadrature, orthogonal polynomials | Burden & Faires; Quarteroni et al. | Ch. 4.7; Ch. 10 |
| Golub–Welsch eigenvalue algorithm | Trefethen & Bau, *Numerical Linear Algebra* | Lecture 37 |
| Comprehensive quadrature theory | Davis & Rabinowitz, *Methods of Numerical Integration* | Chs. 2–4 |
| Clenshaw–Curtis versus Gauss | Trefethen, *ATAP* | Ch. 19 |
| Euler–Maclaurin and periodic trapezoid | Trefethen & Weideman, *SIAM Review* 56 (2014) | entire paper |
| Monte Carlo and quasi-Monte Carlo | Caflisch, *Acta Numerica* 7 (1998) | entire survey |
| Low-discrepancy sequences, Sobol' | Press et al., *Numerical Recipes* | Ch. 7.8–7.9 |
| Variational inference and ELBO integrals | Blei, Kucukelbir & McAuliffe, *JASA* 112 (2017) | entire review |
| Gauss–Hermite in GP classification | Rasmussen & Williams, *GPML* | Ch. 3.4–3.6 |

**Primary references.** Burden & Faires (Ch. 4); Davis & Rabinowitz (1984); Trefethen, *ATAP* (Ch. 19); Quarteroni, Sacco & Saleri (Chs. 9–10); Heath (Ch. 8); Press et al. (Chs. 4, 7); Caflisch (1998); Blei et al. (2017).